<a href="https://colab.research.google.com/github/gopa1959/Machine-Learning-with-Python/blob/master/Copy_of_Session_3_Part_3_Brown_Corpus_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP : Text Classification using POS Tags + Logistic Regression

Reference: https://bekushal.medium.com/fictometer-a-simple-and-explainable-algorithm-for-sentiment-analysis-31186d2a8c7e

**IMPORTANT**

The biggest challenge with analysing **TEXT** data is that computers can only work with numbers, and **TEXT** does not have a natural **numeric representation**. It is UNIQUE among all other data types in this aspect.

In [ ]:
import nltk
from nltk.corpus import brown
nltk.download('brown')
nltk.download('punkt')                     # Tokenizer
nltk.download('punkt_tab')                     # Tokenizer
nltk.download('averaged_perceptron_tagger') # POS Tagger
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')          # Maps tags to ADJ, NOUN, etc.

import pandas as pd
import matplotlib.pyplot as plt

from sklearn import preprocessing
from sklearn import metrics

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# from sklearn.metrics import plot_confusion_matrix
from tqdm.auto import tqdm

# Stanza is another library for text processing.
# For some applications, its better than NLTK.
# import stanza

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


In [ ]:
brown.categories()

['adventure',
 'belles_lettres',
 'editorial',
 'fiction',
 'government',
 'hobbies',
 'humor',
 'learned',
 'lore',
 'mystery',
 'news',
 'religion',
 'reviews',
 'romance',
 'science_fiction']

In [ ]:
brown.fileids(categories='adventure')

['cn01',
 'cn02',
 'cn03',
 'cn04',
 'cn05',
 'cn06',
 'cn07',
 'cn08',
 'cn09',
 'cn10',
 'cn11',
 'cn12',
 'cn13',
 'cn14',
 'cn15',
 'cn16',
 'cn17',
 'cn18',
 'cn19',
 'cn20',
 'cn21',
 'cn22',
 'cn23',
 'cn24',
 'cn25',
 'cn26',
 'cn27',
 'cn28',
 'cn29']

In [ ]:
brown.sents('cn25')

[['Early', 'that', 'day', 'Matsuo', 'saw', 'a', 'marine', '.'], ['The', 'enemy', 'came', 'looming', 'around', 'a', 'bend', 'in', 'the', 'trail', 'and', 'Matsuo', 'took', 'a', 'hasty', 'shot', ',', 'then', 'fled', 'without', 'knowing', 'the', 'result', ',', 'ran', 'until', 'breath', 'was', 'a', 'pain', 'in', 'his', 'chest', 'and', 'his', 'legs', 'were', 'rubbery', '.'], ...]

In [ ]:
brownpostable=pd.DataFrame()

category_list = []
filename_list = []
text_list = []

for i in tqdm(brown.categories()):
    for j in brown.fileids(categories=i):
        # taggedwords=brown.tagged_words(j)
        text = ""
        for sent in brown.sents(j):
            text += " ".join(sent)

        category_list.append(i)
        filename_list.append(j)
        text_list.append(text)

brownpostable["category"] = category_list
brownpostable["filename"] = filename_list
brownpostable["text"] = text_list

brownpostable

  0%|          | 0/15 [00:00<?, ?it/s]

,category,filename,text
0,adventure,cn01,Dan Morgan told himself he would forget Ann Tu...
1,adventure,cn02,Gavin paused wearily .`` You can't stay here w...
2,adventure,cn03,"The sentry was not dead .He was , in fact , sh..."
3,adventure,cn04,`` So it wasn't the earthquake that made him r...
4,adventure,cn05,"She was carrying a quirt , and she started to ..."
...,...,...,...
495,science_fiction,cm02,The expense and time involved are astronomical...
496,science_fiction,cm03,"This was not , for the Angel , just a matter o..."
497,science_fiction,cm04,Ryan hefted his bulk up and supported it on on...
498,science_fiction,cm05,She lived and was given a name .Helva .For her...


# POS Tagging using NLTK

In [ ]:
text = "I am learning about AI Agents from IIT Madras."

words = nltk.word_tokenize(text)
print(words)
print()

tagged_words = nltk.pos_tag(words, tagset='universal')
print(tagged_words)
print()

tag_list = [tag for word, tag in tagged_words]
print(tag_list)

['I', 'am', 'learning', 'about', 'AI', 'Agents', 'from', 'IIT', 'Madras', '.']

[('I', 'PRON'), ('am', 'VERB'), ('learning', 'VERB'), ('about', 'ADP'), ('AI', 'NOUN'), ('Agents', 'NOUN'), ('from', 'ADP'), ('IIT', 'NOUN'), ('Madras', 'NOUN'), ('.', '.')]

['PRON', 'VERB', 'VERB', 'ADP', 'NOUN', 'NOUN', 'ADP', 'NOUN', 'NOUN', '.']


In [ ]:
def pos_count_nltk(row):
    text = row.text

    # 2. Tokenize the text into individual words
    words = nltk.word_tokenize(text)

    # 3. Tag the words and force the Universal tagset
    # This returns a list of tuples like: [('The', 'DET'), ('quick', 'ADJ'), ...]
    tagged_words = nltk.pos_tag(words, tagset='universal')

    # 4. Extract just the tags from the tuples
    tag_list = [tag for word, tag in tagged_words]

    adj_count = tag_list.count('ADJ')
    adv_count = tag_list.count('ADV')
    pron_count = tag_list.count('PRON')
    noun_count = tag_list.count('NOUN')
    verb_count = tag_list.count('VERB')

    return pd.Series([adj_count, adv_count, pron_count, noun_count, verb_count])

# Initialize the progress bar
tqdm.pandas(desc="Tagging Parts of Speech (NLTK)")

# Apply the function
brownpostable[['ADJ', 'ADV', 'PRON', 'NOUN', 'VERB']] = brownpostable.progress_apply(pos_count_nltk, axis=1)

brownpostable

Tagging Parts of Speech (NLTK):   0%|          | 0/500 [00:00<?, ?it/s]

,category,filename,text,ADJ,ADV,PRON,NOUN,VERB
0,adventure,cn01,Dan Morgan told himself he would forget Ann Tu...,128,146,256,452,515
1,adventure,cn02,Gavin paused wearily .`` You can't stay here w...,122,133,251,499,465
2,adventure,cn03,"The sentry was not dead .He was , in fact , sh...",106,87,160,540,477
3,adventure,cn04,`` So it wasn't the earthquake that made him r...,147,98,199,554,454
4,adventure,cn05,"She was carrying a quirt , and she started to ...",125,163,259,450,502
...,...,...,...,...,...,...,...,...
495,science_fiction,cm02,The expense and time involved are astronomical...,162,113,150,547,430
496,science_fiction,cm03,"This was not , for the Angel , just a matter o...",146,159,179,479,446
497,science_fiction,cm04,Ryan hefted his bulk up and supported it on on...,180,147,217,462,456
498,science_fiction,cm05,She lived and was given a name .Helva .For her...,213,110,145,593,387


# Data Preparation and Classification

In [ ]:
# You can use something called RFECV for finding relevant features.

brownpostable["RADJPRON"] = brownpostable["ADJ"]/brownpostable["PRON"]
brownpostable["RADVADJ"] = brownpostable["ADV"]/brownpostable["ADJ"]
brownpostable["RNOUNVERB"] = brownpostable["NOUN"]/brownpostable["VERB"]
brownpostable

,category,filename,text,ADJ,ADV,PRON,NOUN,VERB,RADJPRON,RADVADJ,RNOUNVERB
0,adventure,cn01,Dan Morgan told himself he would forget Ann Tu...,128,146,256,452,515,0.500000,1.140625,0.877670
1,adventure,cn02,Gavin paused wearily .`` You can't stay here w...,122,133,251,499,465,0.486056,1.090164,1.073118
2,adventure,cn03,"The sentry was not dead .He was , in fact , sh...",106,87,160,540,477,0.662500,0.820755,1.132075
3,adventure,cn04,`` So it wasn't the earthquake that made him r...,147,98,199,554,454,0.738693,0.666667,1.220264
4,adventure,cn05,"She was carrying a quirt , and she started to ...",125,163,259,450,502,0.482625,1.304000,0.896414
...,...,...,...,...,...,...,...,...,...,...,...
495,science_fiction,cm02,The expense and time involved are astronomical...,162,113,150,547,430,1.080000,0.697531,1.272093
496,science_fiction,cm03,"This was not , for the Angel , just a matter o...",146,159,179,479,446,0.815642,1.089041,1.073991
497,science_fiction,cm04,Ryan hefted his bulk up and supported it on on...,180,147,217,462,456,0.829493,0.816667,1.013158
498,science_fiction,cm05,She lived and was given a name .Helva .For her...,213,110,145,593,387,1.468966,0.516432,1.532300


In [ ]:
brown2=brownpostable.copy()
for i in ['news','reviews','government','learned','hobbies']:
    brown2=brown2.replace(to_replace=i,value='nonfiction')

for i in ['fiction','mystery','science_fiction','adventure','romance']:
    brown2=brown2.replace(to_replace=i,value='fiction')

index_names=brown2[(brown2['category'] != 'fiction') & (brown2['category'] != 'nonfiction')].index
brown2.drop(index_names,inplace=True)

In [ ]:
brown2.head(10)

,category,filename,text,ADJ,ADV,PRON,NOUN,VERB,RADJPRON,RADVADJ,RNOUNVERB
0,fiction,cn01,Dan Morgan told himself he would forget Ann Tu...,128,146,256,452,515,0.500000,1.140625,0.877670
1,fiction,cn02,Gavin paused wearily .`` You can't stay here w...,122,133,251,499,465,0.486056,1.090164,1.073118
2,fiction,cn03,"The sentry was not dead .He was , in fact , sh...",106,87,160,540,477,0.662500,0.820755,1.132075
3,fiction,cn04,`` So it wasn't the earthquake that made him r...,147,98,199,554,454,0.738693,0.666667,1.220264
4,fiction,cn05,"She was carrying a quirt , and she started to ...",125,163,259,450,502,0.482625,1.304000,0.896414
5,fiction,cn06,Such was my state of mind that I did not quest...,114,143,249,470,433,0.457831,1.254386,1.085450
6,fiction,cn07,"The flat , hard cap was small , but he thrust ...",123,97,187,619,446,0.657754,0.788618,1.387892
7,fiction,cn08,If she sensed any unusual preoccupation on the...,155,131,210,495,424,0.738095,0.845161,1.167453
8,fiction,cn09,"Miraculously , she found exactly the right sta...",149,188,263,437,468,0.566540,1.261745,0.933761
9,fiction,cn10,The Brannon outfit -- known as the Slash-B bec...,123,124,222,546,457,0.554054,1.008130,1.194748


In [ ]:
brown3=brown2.replace(to_replace='nonfiction',value='0')
brown3=brown3.replace(to_replace='fiction',value='1')

In [ ]:

brown3

,category,filename,text,ADJ,ADV,PRON,NOUN,VERB,RADJPRON,RADVADJ,RNOUNVERB
0,1,cn01,Dan Morgan told himself he would forget Ann Tu...,128,146,256,452,515,0.500000,1.140625,0.877670
1,1,cn02,Gavin paused wearily .`` You can't stay here w...,122,133,251,499,465,0.486056,1.090164,1.073118
2,1,cn03,"The sentry was not dead .He was , in fact , sh...",106,87,160,540,477,0.662500,0.820755,1.132075
3,1,cn04,`` So it wasn't the earthquake that made him r...,147,98,199,554,454,0.738693,0.666667,1.220264
4,1,cn05,"She was carrying a quirt , and she started to ...",125,163,259,450,502,0.482625,1.304000,0.896414
...,...,...,...,...,...,...,...,...,...,...,...
495,1,cm02,The expense and time involved are astronomical...,162,113,150,547,430,1.080000,0.697531,1.272093
496,1,cm03,"This was not , for the Angel , just a matter o...",146,159,179,479,446,0.815642,1.089041,1.073991
497,1,cm04,Ryan hefted his bulk up and supported it on on...,180,147,217,462,456,0.829493,0.816667,1.013158
498,1,cm05,She lived and was given a name .Helva .For her...,213,110,145,593,387,1.468966,0.516432,1.532300


In [ ]:
brown3["category"].value_counts()

,count
category,
0,207
1,117


# ADV/ADJ and ADJ/PRON

In [ ]:
x=brown3[["RADJPRON", "RADVADJ"]]
y=brown3.category

In [ ]:
x

,RADJPRON,RADVADJ
0,0.500000,1.140625
1,0.486056,1.090164
2,0.662500,0.820755
3,0.738693,0.666667
4,0.482625,1.304000
...,...,...
495,1.080000,0.697531
496,0.815642,1.089041
497,0.829493,0.816667
498,1.468966,0.516432


In [ ]:
y

,category
0,1
1,1
2,1
3,1
4,1
...,...
495,1
496,1
497,1
498,1


In [ ]:
# GENERALISATION
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
logreg = LogisticRegression(solver='lbfgs')
logreg.fit(x,y)

LogisticRegression()

In [ ]:
y_pred=logreg.predict(x_train)
accuracy = metrics.accuracy_score(y_train,y_pred)
print("Training Accuracy : ",accuracy)

Training Accuracy :  0.9691119691119691


In [ ]:
y_pred=logreg.predict(x_test)
accuracy = metrics.accuracy_score(y_test,y_pred)
print("Testing Accuracy : ", accuracy)

Testing Accuracy :  0.9384615384615385


In [ ]:
confusion_matrix(y_test,y_pred)

array([[36,  0],
       [ 4, 25]])

**DIY:**

1. Do clustering of this dataset using the same two input features (first use k=2 and then check appropriate number of clusters using the elbow method), and measure what fraction of points in each cluster have the same label (fiction vs non-fiction).

2. Try other NLP features for this classification and evaluate their accuracy.

3. Try this approach on other classification datasets (eg. reviews):
https://bekushal.medium.com/curated-machine-learning-datasets-for-college-students-ba8cfbc98b6b
